# MedMCQA + MedQA: deduplicated splits with three MedMCQA evaluation subsets (v6)

One call returns fixed, leak-free splits in one common format:

```python
from medsplits import load_splits, build_messages
splits = load_splits("medmcqa")   # {"train", "hp_dev", "calib", "thresh", "test"}
medqa = load_splits("medqa")      # {"train", "val", "test"}
messages = build_messages(splits["train"][0])
```

**How to use this notebook:** run the cells top to bottom. Cell 1 is the only place that decides where data lives.
The code is written to `/content/proj` (a temporary Colab folder), so copy `medsplits/` and `tests/` into the team repo
when you are happy. The data itself goes to `DATA_DIR` (Google Drive), never into git.

**Three MedMCQA evaluation subsets (v4), made test-like by a similarity filter (v5, threshold 0.85 in v6)**

The MedMCQA `val` split is replaced by three disjoint subsets carved from train, so that tuning, calibration and thresholding never
share questions and none of them touches `test` (the official MedMCQA validation set, used only for final numbers).

| Subset | Nominal size | Used for | Never used for |
|---|---|---|---|
| `hp_dev` | 2000 | choosing learning rate and LoRA rank | fitting T, choosing the threshold, final numbers |
| `calib` | 1500 | fitting the temperature T of each checkpoint | tuning, choosing the threshold, final numbers |
| `thresh` | 2500 | choosing the abstention threshold of each checkpoint | tuning, fitting T, final numbers |
| `test` | official validation set | final reported numbers only | any fitting or selection |

MedQA keeps its single `val` split (1000 rows). Subsets are carved with a fixed seed after sorting ids, so the result does not depend
on load order; a few rows per subset can be dropped by the leak guard, so actual sizes are slightly below nominal (see `stats.json`).

**Why the filter exists.** MedMCQA's authors built its dev/test sets so that questions with a Levenshtein similarity above 0.9 to
other questions were left out. The v4 diagnostic on the real data showed the consequence: 21.8% of random train rows have a different
train question at similarity 0.9 or more, against 0.5% of `test` rows (at 0.8: 34.6% vs 7.4%; median nearest similarity 0.71 vs 0.56).
Random subsets of train would therefore be easier and look more familiar to a trained model than `test`, which biases the fitted
temperature and abstention threshold. A candidate row is only eligible for `hp_dev`, `calib` or `thresh` if no other train
question has similarity at or above `SIMILARITY_FILTER["medmcqa"]["threshold"]` to it (identical text is ignored, the leak guard handles that). Because the comparison is made
against the whole train set before carving, the three subsets are also free of near-copies of each other.

The v5 run used 0.9 and removed the 0.9 gap completely (0.0% vs 0.6% for `test`), but at 0.8 the subsets were still easier than `test`
(hp_dev 20.0%, calib 14.9%, thresh 16.2% vs 8.0%). v6 uses 0.85 as a compromise: it closes more of that gap without pushing the subsets
below `test` the other way. The post-build table shows what the real shares are.

**Two-tier deduplication (from v3)**

| Where | Rule | Why |
|---|---|---|
| Inside train (and across the two train sets) | `exact_key`: same question **and** same options | Removes only true copies. Measured on the reference run: v1's question-only rule removed 30652 MedMCQA train rows, the exact rule removes 18106. The rest were distinct questions. |
| Train vs val/test, and inside val/test, and between the two datasets' eval sets | `leak_key`: long stems (10+ words) match on the question alone; short stems also need the same options | Strict where it matters: a reworded copy of an eval question must not survive in train. |

Earlier fixes kept from v2: pinned dataset commits (recorded in `stats.json`), a config guard in `load_splits`, and tests that check every row.

**Shared settings (all live in `medsplits/data.py`).** Changing any of them changes the splits, so rebuild and re-share `splits/`.

| Setting | Meaning |
|---|---|
| `DATA_DIR` (env var) | Where data, ids and the HF cache are stored. Default `/shared/data`. |
| `SIMILARITY_FILTER` | Per dataset: similarity threshold and candidate-pool factor of the filter above (MedMCQA only). |
| `SEED`, `SPLIT_SIZES` | Shuffle seed and the named subsets carved from each train set (MedMCQA: `hp_dev`, `calib`, `thresh`; MedQA: `val`). |
| `SHORT_STEM_WORDS` | `leak_key` threshold: stems shorter than this also need identical options to match. |
| `DEDUPE_SCHEME` | Label of the dedupe rules (two-tier). Bump it if you change how keys are built. |
| `HF_REPOS`, `HF_REVISIONS` | Which datasets, and which commit of each (pinned below). |
| `SPLIT_PRIORITY`, `DATASET_ORDER` | Which copy survives a duplicate (test > val > hp_dev > calib > thresh > train; MedQA before MedMCQA). |

## Cell 1: environment

Installs the libraries, mounts Drive (only when running on Colab) and sets `DATA_DIR`.
`DATA_DIR` must be set **before** `medsplits` is imported, because `data.py` reads it once at import time.
Every teammate should point it at the same shared folder.

In [1]:
!pip -q install datasets pytest rapidfuzz==3.14.6
import os

DATA_DIR = "/content/drive/MyDrive/shared/data"   # <- the one line to edit per team / machine

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass  # not on Colab: DATA_DIR just has to be a writable folder

os.environ["DATA_DIR"] = DATA_DIR
os.makedirs(DATA_DIR, exist_ok=True)
print("DATA_DIR =", DATA_DIR)

Mounted at /content/drive
DATA_DIR = /content/drive/MyDrive/shared/data


## Cell 2: project folders

Creates the package folder `medsplits/` and the `tests/` folder. The following cells write the Python files with
`%%writefile`; the first one overwrites the file and the rest append (`-a`), so you can re-run the notebook top to
bottom safely.

In [2]:
!mkdir -p /content/proj/medsplits /content/proj/tests

## `data.py`, part 1: imports and shared configuration

Imports only what the module needs, then defines every setting that shapes the splits in one block.
`current_config()` (next cell) snapshots this block, and `build_all` saves the snapshot next to the splits.

- `DATA_DIR`: resolved once at import from the env var, falling back to `/shared/data`.
- `SPLIT_SIZES`: the named subsets carved out of each dataset's train set, in order, with their sizes.
- `SHORT_STEM_WORDS`: threshold used by `leak_key`.
- `DEDUPE_SCHEME`: a label for the two-tier rules; it is part of the saved config, so splits built under another scheme are rejected.
- `HF_REVISIONS`: pinned to the two commits resolved by the first full build. `None` would mean "latest". After a build, `stats.json`
  records `resolved_revisions`, which must equal these pins.
- `SPLIT_PRIORITY` / `DATASET_ORDER`: decide which copy of a duplicate survives (test first, then val, hp_dev, calib, thresh, and
  train last). MedQA goes first so its official test set (the source of MedAbstain) is never trimmed by MedMCQA.
- `RESOLVED_REVISIONS`: filled while loading, so the build can record the exact dataset commits used.

In [3]:
%%writefile /content/proj/medsplits/data.py
"""Load MedMCQA and MedQA, convert both to one format, deduplicate, and build fixed splits.

Entry point: load_splits("medmcqa" | "medqa") -> {split name: [examples]}
MedMCQA: train, hp_dev, calib, thresh, test.  MedQA: train, val, test.
Example format: {"id", "source", "question", "options": {"A": ..., "B": ...}, "answer": "B"}
"""
import hashlib
import importlib.metadata
import json
import math
import os
import random
import re
import string
import sys
from collections import Counter
from pathlib import Path

# -------------------------------------------------------------------------------------------------
# Shared configuration. Everything here defines the splits. If you change any of it, run build_all()
# again and re-share the splits folder: load_splits refuses to run on splits built with another config.
# -------------------------------------------------------------------------------------------------
DATA_DIR = Path(os.environ.get("DATA_DIR", "/shared/data"))  # set the env var BEFORE importing
SEED = 42
SPLIT_SIZES = {  # rows carved out of each dataset's train set, in this order (the rest stays train)
    "medmcqa": {"hp_dev": 2000, "calib": 1500, "thresh": 2500},
    "medqa": {"val": 1000},
}
SIMILARITY_FILTER = {  # subset rows must have no other train question with Levenshtein similarity >= threshold
    "medmcqa": {"threshold": 0.85, "pool_factor": 2},  # pool_factor: candidates examined = pool_factor * total subset rows
}
SHORT_STEM_WORDS = 10  # leak_key: stems with fewer words must also match on options
DEDUPE_SCHEME = "two-tier-v1"  # exact_key inside train, leak_key against val/test; bump if keys change
HF_REPOS = {"medmcqa": "openlifescienceai/medmcqa", "medqa": "GBaker/MedQA-USMLE-4-options"}
HF_REVISIONS = {  # pinned dataset commits (None would mean "latest")
    "medmcqa": "91c6572c454088bf71b679ad90aa8dffcd0d5868",
    "medqa": "0fb93dd23a7339b6dcd27e241cb9b5eca62d4d18",
}
SPLIT_PRIORITY = ("test", "val", "hp_dev", "calib", "thresh", "train")  # a repeated question stays in the earliest; train last
DATASET_ORDER = ("medqa", "medmcqa")  # tie-break inside a split: MedQA first (protects MedAbstain)
LETTERS = string.ascii_uppercase  # option labels by position, so any option count up to 26 works

RESOLVED_REVISIONS = {}  # filled by _load_hf: the dataset commits actually used
_PUNCT = re.compile(r"[^\w\s]")

Writing /content/proj/medsplits/data.py


## `current_config()`

Returns the settings above as a plain dict. `build_all` stores it in `splits/stats.json`; `load_splits` compares the
stored copy with the current one. This is what keeps a team on identical splits: if someone edits `SEED`,
`SPLIT_SIZES`, `SIMILARITY_FILTER`, the threshold, the repos, the revisions or the priority order and forgets to rebuild, loading fails with a
clear message instead of silently using different data.

In [4]:
%%writefile -a /content/proj/medsplits/data.py


def current_config() -> dict:
    """Snapshot of every setting that changes the splits (JSON-serializable)."""
    return {
        "seed": SEED,
        "split_sizes": SPLIT_SIZES,
        "similarity_filter": SIMILARITY_FILTER,
        "short_stem_words": SHORT_STEM_WORDS,
        "dedupe_scheme": DEDUPE_SCHEME,
        "hf_repos": HF_REPOS,
        "hf_revisions": HF_REVISIONS,
        "split_priority": list(SPLIT_PRIORITY),
        "dataset_order": list(DATASET_ORDER),
    }

Appending to /content/proj/medsplits/data.py


## `normalize(text)`

Turns a question into a comparison form: lowercase, every punctuation character replaced by a space, whitespace runs
collapsed. So `"What is X-ray?"` and `"what is x ray"` compare equal. It is deliberately simple. Changing it changes
which rows count as duplicates, so it is part of the shared configuration in practice.

In [5]:
%%writefile -a /content/proj/medsplits/data.py


def normalize(text: str) -> str:
    """Lowercase, replace punctuation with spaces, collapse whitespace."""
    return " ".join(_PUNCT.sub(" ", text.lower()).split())

Appending to /content/proj/medsplits/data.py


## `exact_key(ex)`, `leak_key(ex)` and `_answer_text(ex)` (two-tier keys, new in v3)

Two different questions are asked about "the same question", so there are two keys.

- **`exact_key`**: normalized question + sorted, normalized options. Two rows match only if they are true copies (a
  reordered copy of the same options still matches). Used to deduplicate **train**. On the reference run this is the 18106 figure
  from the sweep: removing only these is safe for training, and it keeps distinct questions that share a generic stem.
- **`leak_key`**: long stems (at least `SHORT_STEM_WORDS` words) match on the normalized question alone, so a clinical
  vignette whose distractors were edited still counts as the same question; short stems also include the sorted options,
  so generic stems like "Which of the following is true?" do not collide. Used for everything that protects **evaluation**:
  val/test against train, inside val/test, and between the two datasets' eval sets.

Why two: wrongly dropping a distinct question from train only costs a little training data, while keeping a reworded copy
of an eval question in train corrupts the evaluation. So the strict key guards the eval sets and the exact key cleans train.

`_answer_text` returns the normalized text of the correct option. `dedupe_within` uses it to count removed copies that
disagree on the answer. Under `exact_key` the options are identical, so a disagreement is genuine label noise.

In [6]:
%%writefile -a /content/proj/medsplits/data.py


def _answer_text(ex: dict) -> str:
    """Normalized text of the correct option (to detect duplicates that disagree on the answer)."""
    return normalize(ex["options"][ex["answer"]])


def _options_key(ex: dict) -> str:
    return " | ".join(sorted(normalize(v) for v in ex["options"].values()))


def exact_key(ex: dict) -> str:
    """True-copy identity: same question and same set of options. Used to deduplicate train."""
    return f"{normalize(ex['question'])} || {_options_key(ex)}"


def leak_key(ex: dict) -> str:
    """Strict identity for protecting val/test: long stems match on the question alone."""
    q = normalize(ex["question"])
    if len(q.split()) >= SHORT_STEM_WORDS:
        return q
    return f"{q} || {_options_key(ex)}"

Appending to /content/proj/medsplits/data.py


## `_load_hf(name)` (new in v2)

One place that downloads a dataset. It passes the pinned `revision` (or latest when `None`), caches into
`DATA_DIR/hf_cache` so the download persists on Drive, and records the commit hash that was actually used in
`RESOLVED_REVISIONS`. If the Hub lookup for the hash fails, it records the requested revision instead and carries on.

In [7]:
%%writefile -a /content/proj/medsplits/data.py


def _load_hf(name: str):
    """Download one dataset (pinned revision if set) and remember which commit was used."""
    from datasets import load_dataset

    rev = HF_REVISIONS[name]
    ds = load_dataset(HF_REPOS[name], revision=rev, cache_dir=str(DATA_DIR / "hf_cache"))
    try:
        from huggingface_hub import HfApi

        RESOLVED_REVISIONS[name] = HfApi().dataset_info(HF_REPOS[name], revision=rev).sha
    except Exception:
        RESOLVED_REVISIONS[name] = rev
    return ds

Appending to /content/proj/medsplits/data.py


## `load_medmcqa_raw()`

Converts MedMCQA to the common format and returns `{"train": [...], "test": [...]}`.

- HF `train` becomes our train. HF `validation` becomes our **test**, because HF `test` has no usable answers.
- `cop` (0 to 3) becomes a letter. Rows without a valid `cop` are skipped (none were skipped in the v1 run).
- `id` is `medmcqa-<original uuid>`, so ids never collide with MedQA.
- `options` is a dict built from `opa` to `opd` by position, in the same shape MedAbstain needs (any number of options).

In [8]:
%%writefile -a /content/proj/medsplits/data.py


def load_medmcqa_raw() -> dict:
    """MedMCQA in the common format. Official validation is used as our test (official test has no answers)."""
    ds = _load_hf("medmcqa")
    out = {}
    for hf_split, ours in (("train", "train"), ("validation", "test")):
        rows = []
        for r in ds[hf_split]:
            if r["cop"] not in (0, 1, 2, 3):
                continue
            opts = [r["opa"], r["opb"], r["opc"], r["opd"]]
            rows.append({
                "id": f"medmcqa-{r['id']}",
                "source": "medmcqa",
                "question": r["question"].strip(),
                "options": {LETTERS[i]: str(o).strip() for i, o in enumerate(opts)},
                "answer": LETTERS[r["cop"]],
            })
        out[ours] = rows
    return out

Appending to /content/proj/medsplits/data.py


## `load_medqa_raw()`

Converts MedQA (English, 4 options) to the common format and returns `{"train": [...], "test": [...]}`.

- HF only has train and test, so validation is carved from train later.
- `options` already arrives as a dict keyed A to D; it is sorted and stripped.
- The `assert` fails loudly if an answer letter is not one of the options.
- MedQA has no ids, so `id` is `medqa-<split>-<row index>`. This is stable as long as the pinned revision (and therefore row order) is unchanged, which is one more reason to pin `HF_REVISIONS`.

In [9]:
%%writefile -a /content/proj/medsplits/data.py


def load_medqa_raw() -> dict:
    """MedQA (English, 4 options) in the common format. HF has train and test only."""
    ds = _load_hf("medqa")
    out = {}
    for split in ("train", "test"):
        rows = []
        for i, r in enumerate(ds[split]):
            opts = {k: str(v).strip() for k, v in sorted(r["options"].items())}
            assert r["answer_idx"] in opts, (split, i)
            rows.append({
                "id": f"medqa-{split}-{i}",
                "source": "medqa",
                "question": r["question"].strip(),
                "options": opts,
                "answer": r["answer_idx"],
            })
        out[split] = rows
    return out

Appending to /content/proj/medsplits/data.py


## `dedupe_within(rows, key_fn)`

Removes repeated rows inside one list, keeping the **first** occurrence as decided by `key_fn` (`exact_key` for train,
`leak_key` for test). Returns `(kept_rows, number_removed, number_of_removed_rows_whose_answer_text_differs_from_the_kept_one)`.

The third number counts removed copies that disagree on the correct answer. With `exact_key` the options are identical, so
this is genuine label noise in the source dataset (it was 1983 under the old question-only key, but that number mixed in
different-options rows). It is reported in `stats.json` and does not change which rows are kept.

In [10]:
%%writefile -a /content/proj/medsplits/data.py


def dedupe_within(rows: list, key_fn):
    """Keep the first row per key_fn(row). Returns (kept, n_removed, n_removed_with_conflicting_answer)."""
    first, kept = {}, []
    removed = conflicts = 0
    for r in rows:
        k = key_fn(r)
        if k not in first:
            first[k] = r
            kept.append(r)
            continue
        removed += 1
        if _answer_text(first[k]) != _answer_text(r):
            conflicts += 1
    return kept, removed, conflicts

Appending to /content/proj/medsplits/data.py


## `split_names(name)` (new in v4)

Returns the split names of one dataset in the order used everywhere (ids files, `load_splits`, stats): `train`, then the carved
subsets in the order they are listed in `SPLIT_SIZES`, then `test`.

- MedMCQA: `train, hp_dev, calib, thresh, test`
- MedQA: `train, val, test`

The order also matters for reproducibility, because the id files are written in this order.

In [11]:
%%writefile -a /content/proj/medsplits/data.py


def split_names(name: str) -> tuple:
    """Split names of one dataset: train, the carved subsets (SPLIT_SIZES order), test."""
    return ("train", *SPLIT_SIZES[name], "test")

Appending to /content/proj/medsplits/data.py


## `near_duplicate_ids(rows, reference, threshold, cache_dir)` (new in v5)

Returns the ids of the `rows` that have a **different** question in `reference` with Levenshtein similarity of at least
`threshold` (computed on `normalize`d question text, 0 to 1). `rapidfuzz.process.cdist` does the comparison in compiled code on
all CPU cores, in chunks of 100 rows so memory stays small. Rows whose normalized text is identical to a reference row are not
flagged (that is a copy, handled by the leak guard), which also stops a row from matching itself.

The result is a pure function of the texts, so it is cached in `cache_dir/similarity_<sha256>.json`; the key hashes the threshold,
the candidate ids and the reference ids. A cache file is only a speed-up: deleting it gives the identical result. `rapidfuzz` is
imported inside the function, so the rest of the module works without it.

In [12]:
%%writefile -a /content/proj/medsplits/data.py


def near_duplicate_ids(rows: list, reference: list, threshold: float, cache_dir=None) -> set:
    """Ids of rows with a different-text question in reference at Levenshtein similarity >= threshold."""
    key = hashlib.sha256("\n".join(
        [f"threshold={threshold}", *(e["id"] for e in rows), "--", *(e["id"] for e in reference)]
    ).encode()).hexdigest()
    cache_file = Path(cache_dir) / f"similarity_{key}.json" if cache_dir else None
    if cache_file is not None and cache_file.exists():
        return set(json.loads(cache_file.read_text()))

    import numpy as np
    from rapidfuzz import process
    from rapidfuzz.distance import Levenshtein

    ref_text = [normalize(e["question"]) for e in reference]
    positions = {}
    for i, t in enumerate(ref_text):
        positions.setdefault(t, []).append(i)
    flagged = []
    for start in range(0, len(rows), 100):
        block = [normalize(e["question"]) for e in rows[start:start + 100]]
        sims = process.cdist(block, ref_text, scorer=Levenshtein.normalized_similarity,
                             score_cutoff=threshold, dtype=np.float32, workers=-1)
        for r, text in enumerate(block):
            sims[r, positions.get(text, [])] = 0.0           # ignore identical text (copies, and the row itself)
            if sims[r].max() >= threshold:
                flagged.append(rows[start + r]["id"])
    if cache_file is not None:
        cache_file.parent.mkdir(parents=True, exist_ok=True)
        cache_file.write_text(json.dumps(sorted(flagged)))
    return set(flagged)

Appending to /content/proj/medsplits/data.py


## `carve_subsets(rows, sizes, seed, flag_fn, pool_factor)` (replaces `carve_val`)

Carves named, disjoint subsets out of the deduplicated train set. `sizes` is an ordered dict such as
`{"hp_dev": 2000, "calib": 1500, "thresh": 2500}`.

1. Sort the rows by `id`, so the result does not depend on the order the dataset arrived in.
2. Shuffle once with a private `random.Random(seed)` (no global RNG state is touched).
3. If `flag_fn` is given (MedMCQA: the similarity filter), call it on the first `pool_factor * sum(sizes)` shuffled rows; it returns
   the ids that must not enter a subset.
4. Walk the shuffled rows from the front, skipping flagged ones: the first 2000 eligible rows are `hp_dev`, the next 1500 are
   `calib`, the next 2500 are `thresh`. Every other row, flagged or not, stays in train.

Every subset is a different block of one shuffled list, so subsets cannot share a row. Without `flag_fn` the first rows of the
shuffle are taken directly, as in v4 (this is how MedQA's `val` stays identical). Returns `(remaining_train, {name: rows})`;
raises `ValueError` for non-positive sizes, if the subsets would use up all of train, or if the pool does not contain enough
eligible rows (raise `pool_factor`).

In [13]:
%%writefile -a /content/proj/medsplits/data.py


def carve_subsets(rows: list, sizes: dict, seed: int = SEED, flag_fn=None, pool_factor: float = 2):
    """Deterministically carve disjoint named subsets from rows. Returns (remaining_train, {name: rows}).

    flag_fn(candidate_rows) -> set of ids that are not allowed to enter any subset (optional)."""
    if any(not isinstance(k, int) or k <= 0 for k in sizes.values()):
        raise ValueError(f"subset sizes must be positive integers: {sizes}")
    total = sum(sizes.values())
    if total >= len(rows):
        raise ValueError(f"subsets need {total} rows but only {len(rows)} are available")
    ordered = sorted(rows, key=lambda e: e["id"])
    random.Random(seed).shuffle(ordered)
    limit, blocked = len(ordered), set()
    if flag_fn is not None:
        limit = min(len(ordered), math.ceil(total * pool_factor))
        blocked = set(flag_fn(ordered[:limit]))
    subsets, pos = {}, 0
    for name, k in sizes.items():
        picked = []
        while len(picked) < k:
            if pos >= limit:
                raise ValueError(f"only {len(picked)} eligible rows for {name} in a pool of {limit}; raise pool_factor")
            if ordered[pos]["id"] not in blocked:
                picked.append(ordered[pos])
            pos += 1
        subsets[name] = picked
    chosen = {e["id"] for part in subsets.values() for e in part}
    return [e for e in ordered if e["id"] not in chosen], subsets

Appending to /content/proj/medsplits/data.py


## `global_dedupe(pools)`

The leakage guard, two-tier, now for any number of eval subsets. `pools` is `{dataset: {split_name: [rows]}}`. Splits are visited in
`SPLIT_PRIORITY` order (`test`, `val`, `hp_dev`, `calib`, `thresh`, then `train`) and datasets in `DATASET_ORDER`; a dataset
without a given split is skipped.

- **Every eval split** (test, val, hp_dev, calib, thresh; both datasets) shares one `eval_seen` set of `leak_key`s. A row whose
  `leak_key` was already seen is dropped. So no question is in two eval splits, a question shared by both test sets stays in MedQA
  test and is removed from MedMCQA test, and MedQA test is never trimmed (MedAbstain's source stays intact). If the same question
  lands in two of the MedMCQA subsets, the higher-priority subset keeps it.
- **Train**: dropped if its `leak_key` is in `eval_seen` (strict leakage guard, counted as `cross`), or if its `exact_key` was
  already seen in the other dataset's train or earlier in this one (a true copy, counted as `cross_dup`). Two train rows that only
  share a long stem but have different options are both kept.

Two `assert`s check that `train` is last in `SPLIT_PRIORITY` (the train rule needs the complete eval key set) and that the split
names in `SPLIT_PRIORITY` are exactly the names present in `pools`, so a new subset can never silently skip the guard.

Side effect: subsets are carved *before* this pass, so each one can end up a few rows smaller than its nominal size; the exact sizes
are in `stats.json`.

In [14]:
%%writefile -a /content/proj/medsplits/data.py


def global_dedupe(pools: dict):
    """Two-tier cross-split pass: leak_key protects every eval split, exact_key cleans train."""
    assert SPLIT_PRIORITY[-1] == "train", "train must be processed after every eval split"
    present = {s for parts in pools.values() for s in parts}
    assert present == set(SPLIT_PRIORITY), f"split names {sorted(present)} do not match SPLIT_PRIORITY"
    eval_seen, train_seen = set(), set()
    final = {name: {} for name in pools}
    removed = Counter({f"{n}/{s}/cross": 0 for n in pools for s in pools[n]})
    removed.update({f"{n}/train/cross_dup": 0 for n in pools})
    for split in SPLIT_PRIORITY:
        for name in DATASET_ORDER:
            if split not in pools[name]:
                continue
            kept = []
            for ex in pools[name][split]:
                lk = leak_key(ex)
                if lk in eval_seen:
                    removed[f"{name}/{split}/cross"] += 1
                    continue
                if split == "train":
                    ek = exact_key(ex)
                    if ek in train_seen:
                        removed[f"{name}/train/cross_dup"] += 1
                        continue
                    train_seen.add(ek)
                else:
                    eval_seen.add(lk)
                kept.append(ex)
            final[name][split] = kept
    return final, removed

Appending to /content/proj/medsplits/data.py


## `build_all(out_dir, verbose)`

Runs the whole pipeline and writes everything to disk. Order matters:

1. load both datasets (common format);
2. `dedupe_within` on each list: train with `exact_key`, test with `leak_key`;
3. `carve_subsets` from the deduplicated train, using `SPLIT_SIZES` (MedMCQA: `hp_dev`, `calib`, `thresh`, with the similarity
   filter from `SIMILARITY_FILTER`; MedQA: `val`, no filter);
4. `global_dedupe` across splits and datasets;
5. write `splits/<dataset>.json` (the id lists per split, the source of truth), `processed/<dataset>.jsonl` (the records) and
   `splits/stats.json` (config, resolved dataset commits, raw and final sizes, removal counts, answer-conflict counts).

Deduplicating train *before* carving prevents one copy of a duplicate landing in a subset and another in train.
`out_dir` defaults to `DATA_DIR`; the determinism test passes temporary folders.

In [15]:
%%writefile -a /content/proj/medsplits/data.py


def _environment() -> dict:
    """Versions that could in principle influence the result; stored in stats.json for comparison across machines."""
    def version(pkg):
        try:
            return importlib.metadata.version(pkg)
        except importlib.metadata.PackageNotFoundError:
            return None
    return {"python": sys.version.split()[0], "datasets": version("datasets"), "rapidfuzz": version("rapidfuzz")}


def build_all(out_dir=None, verbose: bool = True, cache_dir=None) -> dict:
    """Build every split from scratch and write ids, records and stats to out_dir."""
    out_dir = Path(out_dir or DATA_DIR)
    cache_dir = Path(cache_dir or DATA_DIR / "cache")
    (out_dir / "splits").mkdir(parents=True, exist_ok=True)
    (out_dir / "processed").mkdir(parents=True, exist_ok=True)

    raw = {"medmcqa": load_medmcqa_raw(), "medqa": load_medqa_raw()}
    raw_sizes = {n: {s: len(v) for s, v in parts.items()} for n, parts in raw.items()}

    removed, conflicts, pools, similarity_info = Counter(), {}, {}, {}
    for name, parts in raw.items():
        train, n_removed, n_conf = dedupe_within(parts["train"], exact_key)
        removed[f"{name}/train/within"], conflicts[f"{name}/train"] = n_removed, n_conf
        test, n_removed, n_conf = dedupe_within(parts["test"], leak_key)
        removed[f"{name}/test/within"], conflicts[f"{name}/test"] = n_removed, n_conf
        flt = SIMILARITY_FILTER.get(name)
        if flt:
            info = similarity_info[name] = {"threshold": flt["threshold"]}

            def flag_fn(pool, reference=train, thr=flt["threshold"], info=info):
                info["pool"] = len(pool)
                info["flagged_in_pool"] = len(bad := near_duplicate_ids(pool, reference, thr, cache_dir))
                return bad

            train, subsets = carve_subsets(train, SPLIT_SIZES[name], SEED, flag_fn, flt["pool_factor"])
        else:
            train, subsets = carve_subsets(train, SPLIT_SIZES[name], SEED)
        pools[name] = {"train": train, **subsets, "test": test}

    final, cross_removed = global_dedupe(pools)
    removed.update(cross_removed)

    stats = {
        "config": json.loads(json.dumps(current_config())),
        "resolved_revisions": dict(RESOLVED_REVISIONS),
        "environment": _environment(),
        "raw_sizes": raw_sizes,
        "removed": dict(removed),
        "similarity_filter": similarity_info,
        "duplicate_answer_conflicts": conflicts,
        "sizes": {n: {s: len(final[n][s]) for s in split_names(n)} for n in final},
    }

    for name, parts in final.items():
        names = split_names(name)
        ids = {s: [e["id"] for e in parts[s]] for s in names}
        (out_dir / "splits" / f"{name}.json").write_text(json.dumps(ids))
        with open(out_dir / "processed" / f"{name}.jsonl", "w", encoding="utf-8") as f:
            for s in names:
                for e in parts[s]:
                    f.write(json.dumps(e, ensure_ascii=False) + "\n")
    (out_dir / "splits" / "stats.json").write_text(json.dumps(stats, indent=2))
    if verbose:
        print(json.dumps(stats, indent=2))
    return stats

Appending to /content/proj/medsplits/data.py


## `load_splits(name, data_dir, check_config)`: the entry point

`load_splits("medmcqa")` returns `{"train", "hp_dev", "calib", "thresh", "test"}` and `load_splits("medqa")` returns
`{"train", "val", "test"}`; each value is a list of common-format dicts.

- If the saved splits do not exist yet, it builds them first.
- It reads the id lists (the source of truth) and looks the records up in `processed/<name>.jsonl`.
- **Config guard:** the configuration stored with the splits must equal the current one; otherwise it raises
  `RuntimeError` telling you to rebuild or revert. Pass `check_config=False` only for debugging.
- It never re-derives splits from random state, so everyone reading the same folder gets identical data.

In [16]:
%%writefile -a /content/proj/medsplits/data.py


def load_splits(name: str, data_dir=None, check_config: bool = True) -> dict:
    """Return {split name: [examples]} for "medmcqa" or "medqa" (see split_names)."""
    if name not in HF_REPOS:
        raise ValueError(f"unknown dataset {name!r}; expected one of {sorted(HF_REPOS)}")
    data_dir = Path(data_dir or DATA_DIR)
    split_file = data_dir / "splits" / f"{name}.json"
    stats_file = data_dir / "splits" / "stats.json"
    if not split_file.exists() or not stats_file.exists():
        build_all(data_dir)

    if check_config:
        saved = json.loads(stats_file.read_text())["config"]
        now = json.loads(json.dumps(current_config()))
        if saved != now:
            raise RuntimeError(
                "Saved splits were built with a different configuration than the current code.\n"
                f"  saved: {saved}\n  now:   {now}\n"
                "Revert the setting, or run build_all() and share the new splits folder."
            )

    ids = json.loads(split_file.read_text())
    with open(data_dir / "processed" / f"{name}.jsonl", encoding="utf-8") as f:
        by_id = {e["id"]: e for e in map(json.loads, f)}
    return {s: [by_id[i] for i in ids[s]] for s in split_names(name)}

Appending to /content/proj/medsplits/data.py


## `prompts.py`

The single definition of the prompt, imported by the model wrapper (#5) and training (#7).
Edit the wording here only, and bump `PROMPT_VERSION` so results can be traced to a prompt.

- `INSTRUCTION`: the final line; `{letters}` is filled from the example's own options, so 4 or 6 options both work.
- `format_question(ex)`: question, blank line, lettered options in letter order, blank line, instruction. Option order in the
  dict does not matter because options are sorted by letter.
- `build_messages(ex, include_answer=False)`: returns chat messages `[{"role": "user", ...}]`. With
  `include_answer=True` it appends an assistant message containing only the correct letter (for supervised training).

In [17]:
%%writefile /content/proj/medsplits/prompts.py
"""The one and only prompt format. Used by the model wrapper (#5) and training (#7)."""

PROMPT_VERSION = "v1"  # bump on any wording change
INSTRUCTION = "Answer with only the letter ({letters})."


def format_question(ex: dict) -> str:
    """Question, lettered options, and the answer-format instruction as one string."""
    options = dict(sorted(ex["options"].items()))
    lines = [ex["question"].strip(), ""]
    lines += [f"{letter}. {text}" for letter, text in options.items()]
    lines += ["", INSTRUCTION.format(letters=", ".join(options))]
    return "\n".join(lines)


def build_messages(ex: dict, include_answer: bool = False) -> list:
    """Chat messages for one example; add the gold letter as an assistant turn for training."""
    messages = [{"role": "user", "content": format_question(ex)}]
    if include_answer:
        messages.append({"role": "assistant", "content": ex["answer"]})
    return messages

Writing /content/proj/medsplits/prompts.py


## `__init__.py`

Re-exports the public names so callers can write `from medsplits import load_splits, build_messages`.

In [18]:
%%writefile /content/proj/medsplits/__init__.py
from .data import build_all, exact_key, leak_key, load_splits, normalize
from .prompts import PROMPT_VERSION, build_messages, format_question

Writing /content/proj/medsplits/__init__.py


## Tests

**`tests/test_data.py`** (needs the real data)

| Test | What it guarantees |
|---|---|
| `test_split_names` | MedMCQA returns exactly `train, hp_dev, calib, thresh, test`; MedQA returns `train, val, test`. |
| `test_medmcqa_splits_have_no_questions_in_common` | `train`, `hp_dev`, `calib`, `thresh` and `test` share no ids and no `leak_key` (checked for every pair). |
| `test_no_eval_question_in_any_train` | No question of any eval split (both datasets) appears in any train set (the required leakage check). |
| `test_eval_sets_have_no_duplicates` | `leak_key`s are unique across all eval splits of both datasets. |
| `test_train_has_no_exact_duplicates` | `exact_key`s are unique across both train sets. |
| `test_subset_sizes_close_to_nominal` | Each carved subset is within 2% of its nominal size (the shortfall is collision removals). |
| `test_subsets_have_no_near_copy_in_train` | A sample of 300 rows from each of `hp_dev`, `calib`, `thresh` has no different train question at or above the filter threshold (the similarity filter works). |
| `test_ids_unique` | Ids are unique across everything. |
| `test_format_all_rows` | Every row: contiguous letters from `A`, answer is an option, correct `source` and id prefix, non-empty question. |
| `test_medqa_test_is_untouched` | MedQA test equals the official test row for row, so MedAbstain's source is intact. |
| `test_deterministic_in_fresh_processes` | Two builds in two separate Python processes (different `PYTHONHASHSEED`, each with an empty similarity cache so the search is recomputed) give identical id files, and both equal the saved splits in `DATA_DIR`. |
| `test_config_guard` | Changing a setting without rebuilding makes `load_splits` raise. |

**`tests/test_subsets.py`** (offline) checks `carve_subsets` (disjoint, sizes, independent of input order, seed-dependent,
prefix property, bad sizes, flagged rows excluded, pool too small), `near_duplicate_ids` (flags reworded copies, ignores identical
text and unrelated questions, cache gives the same answer) and the guard on subsets (a question in two subsets, or in a subset and in train, is kept once).

**`tests/test_dedupe_logic.py`** (offline) checks the key functions and the two-tier pass on tiny hand-made examples.

**`tests/test_prompts.py`** (offline) checks 4 and 6 option prompts, option-order independence and the message structure.

The data tests use the real splits in `DATA_DIR` (built on first use). The two fresh-process rebuilds make the whole run take
several minutes, because each of the two fresh builds repeats the similarity search.

In [19]:
%%writefile /content/proj/tests/test_data.py
import os
import subprocess
import sys
from pathlib import Path

import pytest

from medsplits import data
from medsplits.data import LETTERS, SPLIT_SIZES, exact_key, leak_key, load_medqa_raw, load_splits, split_names

NAMES = ["medmcqa", "medqa"]
ROOT = Path(__file__).resolve().parents[1]  # the folder that contains medsplits/


def eval_names(name):
    return [s for s in split_names(name) if s != "train"]


@pytest.fixture(scope="module")
def splits():
    return {n: load_splits(n) for n in NAMES}


def test_split_names(splits):
    assert list(splits["medmcqa"]) == ["train", "hp_dev", "calib", "thresh", "test"]
    assert list(splits["medqa"]) == ["train", "val", "test"]


def test_medmcqa_splits_have_no_questions_in_common(splits):
    parts = splits["medmcqa"]
    ids = {s: {e["id"] for e in rows} for s, rows in parts.items()}
    keys = {s: {leak_key(e) for e in rows} for s, rows in parts.items()}
    names = list(parts)
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            assert not ids[a] & ids[b], f"shared ids between {a} and {b}"
            assert not keys[a] & keys[b], f"shared questions between {a} and {b}"


def test_no_eval_question_in_any_train(splits):
    train_keys = {leak_key(e) for n in NAMES for e in splits[n]["train"]}
    for n in NAMES:
        for sp in eval_names(n):
            leaked = [e["id"] for e in splits[n][sp] if leak_key(e) in train_keys]
            assert not leaked, (n, sp, leaked[:5])


def test_eval_sets_have_no_duplicates(splits):
    keys = [leak_key(e) for n in NAMES for sp in eval_names(n) for e in splits[n][sp]]
    assert len(keys) == len(set(keys))


def test_train_has_no_exact_duplicates(splits):
    keys = [exact_key(e) for n in NAMES for e in splits[n]["train"]]
    assert len(keys) == len(set(keys))


def test_subset_sizes_close_to_nominal(splits):
    for n, sizes in SPLIT_SIZES.items():
        for sp, nominal in sizes.items():
            actual = len(splits[n][sp])
            assert 0.98 * nominal <= actual <= nominal, (n, sp, actual, nominal)


def test_subsets_have_no_near_copy_in_train(splits):
    """Sampled check of the similarity filter: no hp_dev/calib/thresh row has a different train question at >= the filter threshold."""
    import random

    import numpy as np
    from rapidfuzz import process
    from rapidfuzz.distance import Levenshtein

    train_text = [data.normalize(e["question"]) for e in splits["medmcqa"]["train"]]
    identical = {}
    for i, t in enumerate(train_text):
        identical.setdefault(t, []).append(i)
    thr = data.SIMILARITY_FILTER["medmcqa"]["threshold"]
    rng = random.Random(0)
    for sp in ("hp_dev", "calib", "thresh"):
        sample = [data.normalize(e["question"]) for e in rng.sample(splits["medmcqa"][sp], min(300, len(splits["medmcqa"][sp])))]
        sims = process.cdist(sample, train_text, scorer=Levenshtein.normalized_similarity,
                             score_cutoff=thr, dtype=np.float32, workers=-1)
        for r, t in enumerate(sample):
            sims[r, identical.get(t, [])] = 0.0
        assert sims.max() < thr, (sp, float(sims.max()))


def test_ids_unique(splits):
    ids = [e["id"] for n in NAMES for sp in split_names(n) for e in splits[n][sp]]
    assert len(ids) == len(set(ids))


def test_format_all_rows(splits):
    for n in NAMES:
        for sp in split_names(n):
            for e in splits[n][sp]:
                letters = list(e["options"])
                assert letters == list(LETTERS[: len(letters)]), e["id"]
                assert len(letters) >= 2, e["id"]
                assert e["answer"] in e["options"], e["id"]
                assert e["source"] == n and e["id"].startswith(n + "-"), e["id"]
                assert e["question"].strip(), e["id"]


def test_medqa_test_is_untouched(splits):
    n_raw = len(load_medqa_raw()["test"])
    assert {e["id"] for e in splits["medqa"]["test"]} == {f"medqa-test-{i}" for i in range(n_raw)}


def _build_in_fresh_process(out_dir, hash_seed):
    """Build in a new Python process with its own EMPTY similarity cache, so the search is really recomputed."""
    env = {**os.environ, "PYTHONHASHSEED": str(hash_seed), "DATA_DIR": str(data.DATA_DIR)}
    code = "import sys; from medsplits.data import build_all; build_all(sys.argv[1], verbose=False, cache_dir=sys.argv[2])"
    subprocess.run([sys.executable, "-c", code, str(out_dir), str(out_dir / "cache")], check=True, env=env, cwd=ROOT)


def test_deterministic_in_fresh_processes(tmp_path, splits):
    _build_in_fresh_process(tmp_path / "a", 0)
    _build_in_fresh_process(tmp_path / "b", 1)
    for n in NAMES:
        a = (tmp_path / "a" / "splits" / f"{n}.json").read_text()
        b = (tmp_path / "b" / "splits" / f"{n}.json").read_text()
        saved = (data.DATA_DIR / "splits" / f"{n}.json").read_text()
        assert a == b == saved, n


def test_config_guard(monkeypatch):
    load_splits("medqa")  # fine with the current config
    monkeypatch.setattr(data, "SEED", data.SEED + 1)
    with pytest.raises(RuntimeError):
        load_splits("medqa")

Writing /content/proj/tests/test_data.py


In [20]:
%%writefile /content/proj/tests/test_subsets.py
import random

import pytest

from medsplits import data
from medsplits.data import carve_subsets, global_dedupe, leak_key

ALL = ("train", "val", "hp_dev", "calib", "thresh", "test")
LONG = "A 45-year-old man presents with chest pain radiating to the left arm and diaphoresis lasting an hour"


def make_rows(n):
    return [
        {"id": f"r{i:04d}", "source": "t",
         "question": f"Question number {i} about a distinct clinical topic and its management in adults",
         "options": {"A": f"a{i}", "B": f"b{i}"}, "answer": "A"}
        for i in range(n)
    ]


def ex(id_, q, opts):
    return {"id": id_, "source": "medmcqa", "question": q, "options": dict(zip("ABCD", opts)), "answer": "A"}


def pools(**kw):
    base = {n: {s: [] for s in ALL} for n in ("medmcqa", "medqa")}
    for key, rows in kw.items():
        name, split = key.split("__")
        base[name][split] = rows
    return base


def ids(rows):
    return [e["id"] for e in rows]


def test_sizes_and_disjoint():
    train, subs = carve_subsets(make_rows(100), {"a": 10, "b": 20, "c": 30})
    assert [len(subs[k]) for k in "abc"] == [10, 20, 30] and len(train) == 40
    everything = ids(train) + [i for k in "abc" for i in ids(subs[k])]
    assert len(everything) == len(set(everything)) == 100


def test_independent_of_input_order():
    rows = make_rows(100)
    shuffled = rows[:]
    random.Random(7).shuffle(shuffled)
    t1, s1 = carve_subsets(rows, {"a": 10, "b": 20})
    t2, s2 = carve_subsets(shuffled, {"a": 10, "b": 20})
    assert ids(t1) == ids(t2) and all(ids(s1[k]) == ids(s2[k]) for k in s1)


def test_deterministic_and_seed_matters():
    rows = make_rows(100)
    assert ids(carve_subsets(rows, {"a": 10}, seed=1)[1]["a"]) == ids(carve_subsets(rows, {"a": 10}, seed=1)[1]["a"])
    assert ids(carve_subsets(rows, {"a": 10}, seed=1)[1]["a"]) != ids(carve_subsets(rows, {"a": 10}, seed=2)[1]["a"])


def test_smaller_carve_is_prefix_of_larger():
    rows = make_rows(100)
    small = ids(carve_subsets(rows, {"v": 10})[1]["v"])
    big = ids(carve_subsets(rows, {"v": 25})[1]["v"])
    assert big[:10] == small
    first = ids(carve_subsets(rows, {"a": 10, "b": 15})[1]["a"])
    assert first == small


def test_rejects_bad_sizes():
    rows = make_rows(20)
    for bad in ({"a": 0}, {"a": -1}, {"a": 20}, {"a": 10, "b": 10}):
        with pytest.raises(ValueError):
            carve_subsets(rows, bad)


def test_question_in_two_subsets_and_train_is_kept_once():
    hp = ex("1", LONG, ["w", "x", "y", "z"])
    cal = ex("2", LONG, ["p", "q", "r", "s"])      # same long stem, different options
    thr = ex("3", LONG.upper(), ["a", "b", "c", "d"])
    trn = ex("4", LONG + "?", ["e", "f", "g", "h"])
    other = ex("5", "A completely different question about renal physiology and fluid balance", ["a", "b", "c", "d"])
    final, removed = global_dedupe(pools(medmcqa__hp_dev=[hp], medmcqa__calib=[cal],
                                         medmcqa__thresh=[thr], medmcqa__train=[trn, other]))
    parts = final["medmcqa"]
    assert ids(parts["hp_dev"]) == ["1"] and parts["calib"] == [] and parts["thresh"] == []
    assert ids(parts["train"]) == ["5"]
    assert removed["medmcqa/calib/cross"] == 1 and removed["medmcqa/thresh/cross"] == 1
    assert removed["medmcqa/train/cross"] == 1


def test_flagged_rows_never_enter_subsets():
    rows = make_rows(200)
    ordered = sorted(rows, key=lambda e: e["id"])
    random.Random(data.SEED).shuffle(ordered)
    blocked = {e["id"] for e in ordered[:15]}               # block the first rows of the shuffle
    train, subs = carve_subsets(rows, {"a": 10, "b": 20}, flag_fn=lambda pool: blocked, pool_factor=3)
    chosen = {e["id"] for part in subs.values() for e in part}
    assert len(subs["a"]) == 10 and len(subs["b"]) == 20 and not chosen & blocked
    assert blocked <= {e["id"] for e in train} and len(train) == 170


def test_pool_too_small_raises():
    with pytest.raises(ValueError):
        carve_subsets(make_rows(100), {"a": 10}, flag_fn=lambda pool: {e["id"] for e in pool}, pool_factor=2)


def test_flag_fn_receives_only_the_pool():
    seen = []
    carve_subsets(make_rows(100), {"a": 10}, flag_fn=lambda pool: seen.append(len(pool)) or set(), pool_factor=2)
    assert seen == [20]


def test_near_duplicate_ids(tmp_path):
    base = "A 45-year-old man presents with chest pain radiating to the left arm and diaphoresis lasting an hour"
    ref = [ex("r1", base, ["a", "b"]), ex("r2", "Completely unrelated question about renal tubular physiology in adults", ["a", "b"])]
    rows = [ex("n1", base.replace("hour", "hours"), ["a", "b"]),          # one character different: flagged
            ex("n2", base.upper() + "?", ["a", "b"]),                      # identical after normalize: not flagged
            ex("n3", "Which vitamin deficiency causes scurvy in sailors on long voyages", ["a", "b"])]
    first = data.near_duplicate_ids(rows, ref, 0.9, tmp_path)
    assert first == {"n1"}
    assert list(tmp_path.glob("similarity_*.json"))
    assert data.near_duplicate_ids(rows, ref, 0.9, tmp_path) == first      # served from the cache
    assert data.near_duplicate_ids(rows, ref, 0.9) == first                # recomputed without a cache


def test_priority_names_match_split_sizes():
    names = {"train", "test"} | {s for sizes in data.SPLIT_SIZES.values() for s in sizes}
    assert names == set(data.SPLIT_PRIORITY) and data.SPLIT_PRIORITY[-1] == "train"

Writing /content/proj/tests/test_subsets.py


In [21]:
%%writefile /content/proj/tests/test_dedupe_logic.py
from medsplits import data
from medsplits.data import dedupe_within, exact_key, global_dedupe, leak_key

LONG = "A 45-year-old man presents with chest pain radiating to the left arm and diaphoresis lasting an hour"
ALL = ("train", "val", "hp_dev", "calib", "thresh", "test")


def ex(id_, q, opts, ans="A", source="medmcqa"):
    return {"id": id_, "source": source, "question": q,
            "options": dict(zip("ABCDEF", opts)), "answer": ans}


def pools(**kw):
    base = {n: {s: [] for s in ALL} for n in ("medmcqa", "medqa")}
    for key, rows in kw.items():
        name, split = key.split("__")
        base[name][split] = rows
    return base


def test_keys():
    a = ex("1", LONG, ["w", "x", "y", "z"])
    b = ex("2", LONG.upper() + "?", ["z", "y", "x", "w"])          # punctuation, case, option order
    c = ex("3", LONG, ["p", "q", "r", "s"])                         # same stem, different options
    assert exact_key(a) == exact_key(b) and exact_key(a) != exact_key(c)
    assert leak_key(a) == leak_key(b) == leak_key(c)                # long stem: options ignored
    s1, s2 = ex("4", "Which is true?", ["a", "b", "c", "d"]), ex("5", "Which is true?", ["e", "f", "g", "h"])
    assert leak_key(s1) != leak_key(s2)                             # short stem: options matter


def test_train_keeps_same_stem_with_different_options():
    a, c = ex("1", LONG, ["w", "x", "y", "z"]), ex("3", LONG, ["p", "q", "r", "s"])
    final, removed = global_dedupe(pools(medmcqa__train=[a, c]))
    assert [e["id"] for e in final["medmcqa"]["train"]] == ["1", "3"]
    assert sum(removed.values()) == 0


def test_reworded_eval_question_is_removed_from_train():
    train = [ex("1", LONG, ["p", "q", "r", "s"]), ex("2", "Another stem entirely about renal physiology and fluid balance in adults", ["a", "b", "c", "d"])]
    test = [ex("9", LONG, ["w", "x", "y", "z"])]                    # same stem, different options
    final, removed = global_dedupe(pools(medmcqa__train=train, medmcqa__test=test))
    assert [e["id"] for e in final["medmcqa"]["train"]] == ["2"]
    assert removed["medmcqa/train/cross"] == 1


def test_true_copies_across_train_sets_are_removed():
    a = ex("1", LONG, ["w", "x", "y", "z"], source="medqa")
    b = ex("2", LONG, ["z", "y", "x", "w"])
    final, removed = global_dedupe(pools(medqa__train=[a], medmcqa__train=[b]))
    assert len(final["medqa"]["train"]) == 1 and len(final["medmcqa"]["train"]) == 0
    assert removed["medmcqa/train/cross_dup"] == 1


def test_medqa_test_wins_over_medmcqa_test():
    q = ex("q", LONG, ["a", "b", "c", "d"], source="medqa")
    m = ex("m", LONG, ["a", "b", "c", "d"])
    final, removed = global_dedupe(pools(medqa__test=[q], medmcqa__test=[m]))
    assert len(final["medqa"]["test"]) == 1 and len(final["medmcqa"]["test"]) == 0
    assert removed["medmcqa/test/cross"] == 1


def test_dedupe_within_counts_conflicts():
    a = ex("1", LONG, ["w", "x", "y", "z"], "A")
    b = ex("2", LONG, ["w", "x", "y", "z"], "B")                    # exact copy, different answer
    kept, removed, conflicts = dedupe_within([a, b], exact_key)
    assert [e["id"] for e in kept] == ["1"] and removed == 1 and conflicts == 1


def test_train_must_be_last_in_priority(monkeypatch):
    monkeypatch.setattr(data, "SPLIT_PRIORITY", ("train", "val", "test"))
    try:
        global_dedupe(pools())
    except AssertionError:
        return
    raise AssertionError("expected the priority assertion to fire")

Writing /content/proj/tests/test_dedupe_logic.py


In [22]:
%%writefile /content/proj/tests/test_prompts.py
from medsplits import build_messages, format_question

SIX = {
    "id": "x-1", "source": "x", "question": "Which drug?",
    "options": {"A": "a", "B": "b", "C": "c", "D": "d", "E": "e", "F": "f"}, "answer": "E",
}
FOUR = {**SIX, "options": {k: v for k, v in SIX["options"].items() if k in "ABCD"}, "answer": "B"}


def test_four_and_six_options():
    assert format_question(FOUR).endswith("Answer with only the letter (A, B, C, D).")
    six = format_question(SIX)
    assert "F. f" in six and six.endswith("(A, B, C, D, E, F).")


def test_option_order_does_not_matter():
    shuffled = {**SIX, "options": dict(reversed(list(SIX["options"].items())))}
    assert format_question(shuffled) == format_question(SIX)


def test_messages():
    assert build_messages(FOUR) == [{"role": "user", "content": format_question(FOUR)}]
    assert build_messages(FOUR, include_answer=True)[-1] == {"role": "assistant", "content": "B"}

Writing /content/proj/tests/test_prompts.py


## Cell: build the splits

`build_all()` downloads both datasets (about 90 MB, cached under `DATA_DIR/hf_cache`), builds the splits and prints the
statistics. What to expect (same pinned commits as v3):

- MedMCQA: `hp_dev`, `calib` and `thresh` close to 2000 / 1500 / 2500 (a handful of rows lost to the leak guard), `test` 4168, and
  train about 158.5k rows (v3 train 160,536 minus the 2000 extra rows moved into the subsets, minus a few more `cross` removals).
- MedQA: 9178 / 1000 / 1273 with MedQA test intact (a few train rows can drop if they match the larger set of MedMCQA subsets;
  `medqa/train/cross` shows it).
- `resolved_revisions` must equal the pins in `HF_REVISIONS`. If it does not, the pin was not applied.

In [23]:
import os, sys
sys.path.insert(0, "/content/proj")

from medsplits import build_all, load_splits, build_messages
stats = build_all()

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

README.md:   0%|          | 0.00/654 [00:00<?, ?B/s]

{
  "config": {
    "seed": 42,
    "split_sizes": {
      "medmcqa": {
        "hp_dev": 2000,
        "calib": 1500,
        "thresh": 2500
      },
      "medqa": {
        "val": 1000
      }
    },
    "similarity_filter": {
      "medmcqa": {
        "threshold": 0.85,
        "pool_factor": 2
      }
    },
    "short_stem_words": 10,
    "dedupe_scheme": "two-tier-v1",
    "hf_repos": {
      "medmcqa": "openlifescienceai/medmcqa",
      "medqa": "GBaker/MedQA-USMLE-4-options"
    },
    "hf_revisions": {
      "medmcqa": "91c6572c454088bf71b679ad90aa8dffcd0d5868",
      "medqa": "0fb93dd23a7339b6dcd27e241cb9b5eca62d4d18"
    },
    "split_priority": [
      "test",
      "val",
      "hp_dev",
      "calib",
      "thresh",
      "train"
    ],
    "dataset_order": [
      "medqa",
      "medmcqa"
    ]
  },
  "resolved_revisions": {
    "medmcqa": "91c6572c454088bf71b679ad90aa8dffcd0d5868",
    "medqa": "0fb93dd23a7339b6dcd27e241cb9b5eca62d4d18"
  },
  "environment": {
    "p

## Cell: look at the result

Prints the split sizes for both datasets and one example (from `calib`) in the common format together with the exact prompt the model will see.

In [24]:
for name in ("medmcqa", "medqa"):
    s = load_splits(name)
    print(name, {k: len(v) for k, v in s.items()})

example = load_splits("medmcqa")["calib"][0]
print(example)
print()
for m in build_messages(example):
    print(f"[{m['role']}]\n{m['content']}")

medmcqa {'train': 158510, 'hp_dev': 1998, 'calib': 1499, 'thresh': 2497, 'test': 4168}
medqa {'train': 9178, 'val': 1000, 'test': 1273}
{'id': 'medmcqa-99beb3e3-ae3e-4ec0-a837-740519aae03d', 'source': 'medmcqa', 'question': "All of the following are true about 'secondary apnea' in a neonate EXCEPT:", 'options': {'A': 'The baby is not breathing', 'B': 'HR may become < 100/min', 'C': 'It may be present even at bih', 'D': 'Baby may respond to tactile stimulation'}, 'answer': 'D'}

[user]
All of the following are true about 'secondary apnea' in a neonate EXCEPT:

A. The baby is not breathing
B. HR may become < 100/min
C. It may be present even at bih
D. Baby may respond to tactile stimulation

Answer with only the letter (A, B, C, D).


## Cell: run the tests

Runs both test files. All tests must pass before the splits are shared. Takes two to three minutes (it rebuilds the splits in two fresh Python processes).

In [25]:
!cd /content/proj && DATA_DIR={DATA_DIR} python -m pytest -q tests

.................................                                        [100%]
33 passed in 486.97s (0:08:06)


## Cell: reference sweep, how many MedMCQA train rows each key removes

This cell only explains the design; the pipeline does not use it. On the reference run it printed 30652 (question only, v1), 27165,
23672, 21542 (threshold 10, v2), 20168, 19226 and 18106 (question + options, now `exact_key` for train). Rerun it any time
to see the same numbers.

It compares how many MedMCQA train rows would be removed under different keys:

- threshold `0`: question text only (the v1 rule, expected to reproduce 30652);
- thresholds `6` to `15`: question-only for stems at least that many words long, question + options for shorter stems;
- threshold `1000000`: question + options for everything (exact duplicates only).

The second output shows where the question-only removals come from, by stem length. Reading it:

- The drop from 30652 to 18106 is distinct questions that share a stem but have different options; v1 wrongly discarded them.
- The remaining 18106 are true copies, about 10% of MedMCQA train.
- The gap between 21542 (threshold 10) and 18106 is long stems with different options. In train these are now kept (two-tier);
  against val/test they are still treated as leaks (the strict `leak_key`).
- The most repeated short stems are printed so you can eyeball them.

In [26]:
from collections import Counter, defaultdict
from medsplits.data import load_medmcqa_raw, normalize

rows = load_medmcqa_raw()["train"]

def n_removed(threshold):
    seen, n = set(), 0
    for e in rows:
        q = normalize(e["question"])
        key = q if len(q.split()) >= threshold else q + " || " + " | ".join(sorted(normalize(v) for v in e["options"].values()))
        if key in seen:
            n += 1
        else:
            seen.add(key)
    return n

print("threshold -> rows removed (0 = question-only, 1000000 = question+options)")
for t in (0, 6, 8, 10, 12, 15, 1_000_000):
    print(f"{t:>8} -> {n_removed(t)}")

groups = defaultdict(list)
for e in rows:
    groups[normalize(e["question"])].append(e)
by_len = Counter()
for q, g in groups.items():
    w = len(q.split())
    bucket = "<5 words" if w < 5 else "5-9 words" if w < 10 else "10-19 words" if w < 20 else "20+ words"
    by_len[bucket] += len(g) - 1
print("\nrows removed by question-only rule, by stem length:", dict(by_len))

print("\nmost repeated stems with fewer than 10 words:")
short = sorted(((len(g), q) for q, g in groups.items() if len(q.split()) < 10), reverse=True)[:8]
for count, q in short:
    n_opt_sets = len({tuple(sorted(normalize(v) for v in e["options"].values())) for e in groups[q]})
    print(f"  x{count} ({n_opt_sets} distinct option sets): {q}")

threshold -> rows removed (0 = question-only, 1000000 = question+options)
       0 -> 30652
       6 -> 27165
       8 -> 23672
      10 -> 21542
      12 -> 20168
      15 -> 19226
 1000000 -> 18106

rows removed by question-only rule, by stem length: {'10-19 words': 8000, '5-9 words': 17312, '20+ words': 1826, '<5 words': 3514}

most repeated stems with fewer than 10 words:
  x13 (11 distinct option sets): call exner bodies are seen in
  x8 (5 distinct option sets): schiller duval bodies are seen in
  x8 (6 distinct option sets): hair on end appearance is seen in
  x8 (8 distinct option sets): all of the following statements are true except
  x8 (7 distinct option sets): all are true except
  x7 (2 distinct option sets): whip lash injury is caused due to
  x7 (6 distinct option sets): which of the following is not a prodrug
  x7 (4 distinct option sets): the term id was coined by


## Cell: how different is `test` from train? (near-duplicate check, run before tuning)

MedMCQA's official split is by exam, and its authors built the dev/test sets to differ from train (questions with a Levenshtein
similarity above 0.9 to others were excluded). Our `test` set is the official validation set, so it has this property. The three
new subsets are random slices of train, which still contain many near-copies of other train questions.

If so, a model would look more accurate and more confident on `hp_dev`, `calib` and `thresh` than on `test`, and the temperature and
abstention threshold fitted there would transfer worse than expected. This cell measures it:

- it takes 1000 random train rows and 1000 `test` rows (fixed seed 0);
- for each, it finds the most similar question in train (Levenshtein similarity, 0 to 1; values below 0.5 are reported as 0);
- neighbours whose normalized text is **identical** are ignored, because the leak guard already removes those from train.

Reading the first line (share with a neighbour at 0.9 or more), as a rule of thumb:

- `test` should be close to 0% by construction.
- If random train rows are under about 2%, the shift is negligible: keep the subsets as they are.
- If they are above about 10%, the subsets are easier than `test`: add a similarity filter before freezing (drop candidate subset
  rows that have a train neighbour at 0.9 or more).
- In between, send the output and we decide together.

It uses `rapidfuzz` and takes a few minutes on Colab. It does not change any splits.

**Reference run (before the v5 filter):** 21.8% of random train rows vs 0.5% of `test` rows had a neighbour at 0.9 or more, so
the filter was added. Because this cell looks at the deduplicated train set before carving, it still shows the unfiltered baseline in v5.

In [27]:
!pip -q install rapidfuzz==3.14.6
import random
from collections import defaultdict

import numpy as np
from rapidfuzz import process
from rapidfuzz.distance import Levenshtein
from medsplits.data import dedupe_within, exact_key, load_medmcqa_raw, normalize

raw = load_medmcqa_raw()
train_rows, _, _ = dedupe_within(raw["train"], exact_key)   # train as the pipeline sees it before carving
train_q = [normalize(e["question"]) for e in train_rows]
test_q = [normalize(e["question"]) for e in raw["test"]]    # official validation = our test

same_text = defaultdict(list)                               # normalized text -> positions in train_q
for i, q in enumerate(train_q):
    same_text[q].append(i)


def nearest_similarity(queries, choices, chunk=100):
    """Highest Levenshtein similarity (0 to 1) of each query to any choice with different text (below 0.5 reported as 0)."""
    best = np.zeros(len(queries), dtype=np.float32)
    for start in range(0, len(queries), chunk):
        block = queries[start:start + chunk]
        sims = process.cdist(block, choices, scorer=Levenshtein.normalized_similarity,
                             score_cutoff=0.5, dtype=np.float32, workers=-1)
        for r, q in enumerate(block):
            sims[r, same_text.get(q, [])] = 0.0             # ignore identical text (includes the row itself)
        best[start:start + chunk] = sims.max(axis=1)
    return best


rng = random.Random(0)
n = 1000
train_sample = rng.sample(train_q, n)
test_sample = rng.sample(test_q, min(n, len(test_q)))
train_best = nearest_similarity(train_sample, train_q)
test_best = nearest_similarity(test_sample, train_q)

print(f"{'share with a train neighbour at least':<40}{'random train rows':>18}{'test rows':>12}")
for t in (0.9, 0.8, 0.7):
    print(f"{t:<40}{(train_best >= t).mean():>18.1%}{(test_best >= t).mean():>12.1%}")
print(f"{'median nearest similarity (0 = below 0.5)':<40}{np.median(train_best):>18.2f}{np.median(test_best):>12.2f}")

share with a train neighbour at least    random train rows   test rows
0.9                                                  21.8%        0.5%
0.8                                                  34.6%        7.4%
0.7                                                  51.5%       22.6%
median nearest similarity (0 = below 0.5)              0.71        0.56


## Cell: did the similarity filter work? (after the build)

Repeats the measurement on the **final** subsets: 1000 random rows from each of `hp_dev`, `calib`, `thresh` (or all rows if fewer)
and 1000 rows of `test`, each compared with the final train set (identical text ignored). The three subsets should now look like
`test`: 0% at and above the filter threshold (guaranteed by construction, which the sampled test also checks) and, as the real
indicator that the shift is reduced, similar shares at 0.8 and 0.7. A small remaining gap below the threshold is expected.

In [28]:
from medsplits import load_splits

final = load_splits("medmcqa")
final_train = [normalize(e["question"]) for e in final["train"]]
same_text = defaultdict(list)
for i, q in enumerate(final_train):
    same_text[q].append(i)

rows_out = {}
for sp in ("hp_dev", "calib", "thresh", "test"):
    qs = [normalize(e["question"]) for e in final[sp]]
    qs = random.Random(0).sample(qs, min(1000, len(qs)))
    rows_out[sp] = nearest_similarity(qs, final_train)

print(f"{'share with a train neighbour at least':<40}" + "".join(f"{sp:>10}" for sp in rows_out))
for t in (0.9, 0.85, 0.8, 0.7):
    print(f"{t:<40}" + "".join(f"{(v >= t).mean():>10.1%}" for v in rows_out.values()))
print(f"{'median nearest similarity':<40}" + "".join(f"{np.median(v):>10.2f}" for v in rows_out.values()))

share with a train neighbour at least       hp_dev     calib    thresh      test
0.9                                           0.0%      0.0%      0.0%      0.6%
0.85                                          0.3%      0.0%      0.2%      2.7%
0.8                                          11.1%      8.2%     10.0%      8.0%
0.7                                          33.9%     31.5%     31.9%     22.5%
median nearest similarity                     0.64      0.63      0.63      0.56


## Cell: numbers for the README

Prints a Markdown table of the final size of every split of both datasets (with the raw sizes), the removed-duplicate counts,
the number of removed copies whose answer disagreed with the kept copy, and the dataset commits used. Paste the output into the
project README (the "Done when" item).

In [29]:
import json
from pathlib import Path

s = json.loads((Path(os.environ["DATA_DIR"]) / "splits" / "stats.json").read_text())
print("| dataset | split | rows |")
print("|---|---|---|")
for n, d in s["sizes"].items():
    for split, rows in d.items():
        print(f"| {n} | {split} | {rows} |")
print("\nRaw sizes (before any removal):", s["raw_sizes"])
print("\nRemoved duplicates (within = copies inside one list; cross = matched an eval question; cross_dup = true copy across the two train sets):")
for k, v in s["removed"].items():
    print(f"- {k}: {v}")
print("\nRemoved duplicates whose answer text disagreed with the kept copy:", s["duplicate_answer_conflicts"])
print("Dataset commits used:", s["resolved_revisions"])
print("Seed:", s["config"]["seed"], "| split sizes:", s["config"]["split_sizes"], "| short-stem threshold:", s["config"]["short_stem_words"])

| dataset | split | rows |
|---|---|---|
| medmcqa | train | 158510 |
| medmcqa | hp_dev | 1998 |
| medmcqa | calib | 1499 |
| medmcqa | thresh | 2497 |
| medmcqa | test | 4168 |
| medqa | train | 9178 |
| medqa | val | 1000 |
| medqa | test | 1273 |

Raw sizes (before any removal): {'medmcqa': {'train': 182822, 'test': 4183}, 'medqa': {'train': 10178, 'test': 1273}}

Removed duplicates (within = copies inside one list; cross = matched an eval question; cross_dup = true copy across the two train sets):
- medmcqa/train/within: 18106
- medmcqa/test/within: 15
- medqa/train/within: 0
- medqa/test/within: 0
- medmcqa/train/cross: 206
- medmcqa/hp_dev/cross: 2
- medmcqa/calib/cross: 1
- medmcqa/thresh/cross: 3
- medmcqa/test/cross: 0
- medqa/train/cross: 0
- medqa/val/cross: 0
- medqa/test/cross: 0
- medmcqa/train/cross_dup: 0
- medqa/train/cross_dup: 0

Removed duplicates whose answer text disagreed with the kept copy: {'medmcqa/train': 400, 'medmcqa/test': 2, 'medqa/train': 0, 'medqa/test

## Cell: checksums of the id files

Prints a SHA-256 checksum of each dataset's id file. Equal checksums mean equal splits, so record them in the README and compare
them after any rebuild or on another machine. The MedQA line is also compared with the v3 reference value: MedQA's carving did
not change in v4, so it should be unchanged unless some MedQA train rows matched the larger set of MedMCQA subsets (see
`medqa/train/cross` above; if that count is 0, the file must be identical).

In [30]:
import hashlib, json, os, pathlib

V3_MEDQA = "6e755d5bbd8c5c256a86e701a0557e1cb8d81e984f6abf7f6d6f57c550ffbc2c"
digests = {}
for p in sorted((pathlib.Path(os.environ["DATA_DIR"]) / "splits").glob("*.json")):
    if p.name != "stats.json":
        digests[p.name] = hashlib.sha256(p.read_bytes()).hexdigest()
        print(p.name, digests[p.name])
print("medqa.json identical to v3:", digests["medqa.json"] == V3_MEDQA)
print("environment of this build:", json.loads((pathlib.Path(os.environ["DATA_DIR"]) / "splits" / "stats.json").read_text())["environment"])

medmcqa.json 7bc0abd2202114ea25c13b3d11aafde2fc9c7469db261042eaa4e5c1764ebaad
medqa.json 6e755d5bbd8c5c256a86e701a0557e1cb8d81e984f6abf7f6d6f57c550ffbc2c
medqa.json identical to v3: True
environment of this build: {'python': '3.13.18', 'datasets': '4.8.5', 'rapidfuzz': '3.14.6'}


## Before sharing

1. Copy `medsplits/` and `tests/` from `/content/proj` into the team repo (Colab's `/content` is wiped when the session ends).
2. The dataset commits are already pinned in `HF_REVISIONS`; confirm `resolved_revisions` in `stats.json` equals them.
3. Share the `DATA_DIR` folder (it holds `splits/`, `processed/` and `hf_cache/`) and add it to `.gitignore`; commit only the code.
4. Everyone sets `DATA_DIR` to that folder before importing `medsplits`.
5. Freeze before any tuning: record the checksums from the checksum cell in the README and do not rebuild afterwards. A rebuild can change ids, which would invalidate runs that already used the old subsets.